In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from geckoAPI.assetsRoster import ROSTER
import warnings

warnings.filterwarnings("ignore")

def compute_momentum_regime(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute momentum-based regime classification
    """
    # Compute log returns
    returns = np.log(df).diff()
    
    # Compute momentum signals
    fast_momentum = returns.rolling(window=30).sum()  # 1-month momentum
    slow_momentum = returns.rolling(window=365).sum() # 12-month momentum
    
    # Convert to binary signals (+1 for positive, -1 for negative)
    fast_signal = np.sign(fast_momentum)
    slow_signal = np.sign(slow_momentum)
    
    # Initialize regime DataFrame
    regime = pd.DataFrame(index=df.index, columns=['regime'])
    
    # Classify regimes
    conditions = [
        (fast_signal == 1) & (slow_signal == 1),   # Bull
        (fast_signal == -1) & (slow_signal == 1),  # Correction
        (fast_signal == -1) & (slow_signal == -1), # Bear
        (fast_signal == 1) & (slow_signal == -1)   # Rebound
    ]
    
    choices = ['bull', 'correction', 'bear', 'rebound']
    regime['regime'] = np.select(conditions, choices, default='undefined')
    
    return regime

def analyze_and_plot_regimes():
    """
    Analyze and plot regime distributions
    """
    all_regimes = {}
    
    # Process each asset
    for asset in ROSTER:
        try:
            # Load and clean data
            df = pd.read_csv(f'micro/assetData/{asset}.csv')
            df['date'] = pd.to_datetime(df['date'])
            
            # Keep only the latest value for each date
            df = df.sort_values('date').groupby('date').last().reset_index()
            
            df.set_index('date', inplace=True)
            
            # Compute regime
            regime_df = compute_momentum_regime(df.iloc[:, 0])
            all_regimes[asset] = regime_df['regime']
            
        except Exception as e:
            print(f"Error processing {asset}: {str(e)}")
    
    # Combine all regimes and ensure index alignment
    combined_regimes = pd.DataFrame(all_regimes)
    combined_regimes = combined_regimes.loc[~combined_regimes.index.duplicated(keep='last')]
    
    # Create plots with proper specs
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Regime Distribution Over Time',
            'Current Regime Distribution',
            'Regime Distribution by Asset',
            'Regime Transitions'
        ),
        specs=[[{"type": "xy"}, {"type": "domain"}],
               [{"type": "xy"}, {"type": "xy"}]]
    )
    
    # Define colors
    colors = {
        'bull': '#03C04A',     # Green
        'correction': '#FFA500', # Orange
        'bear': '#FF0000',      # Red
        'rebound': '#0492C2'    # Blue
    }
    
    # 1. Regime Distribution Over Time
    regime_counts = pd.DataFrame()
    for col in combined_regimes.columns:
        counts = pd.Series(combined_regimes[col]).value_counts()
        regime_counts[col] = counts
    
    regime_percentages = regime_counts.div(regime_counts.sum(axis=0), axis=1) * 100
    
    for regime in ['bull', 'correction', 'bear', 'rebound']:
        if regime in regime_percentages.index:
            fig.add_trace(
                go.Scatter(
                    x=regime_percentages.columns,
                    y=regime_percentages.loc[regime],
                    name=regime,
                    mode='lines',
                    line=dict(color=colors[regime]),
                    stackgroup='one'
                ),
                row=1, col=1
            )
    
    # 2. Current Regime Distribution (Pie Chart)
    latest_regimes = pd.Series(combined_regimes.iloc[-1]).value_counts()
    fig.add_trace(
        go.Pie(
            labels=latest_regimes.index,
            values=latest_regimes.values,
            marker_colors=[colors.get(r, '#GRAY') for r in latest_regimes.index]
        ),
        row=1, col=2
    )
    
    # 3. Regime Distribution by Asset (Box Plot)
    asset_regime_pcts = pd.DataFrame()
    for asset in combined_regimes.columns:
        counts = pd.Series(combined_regimes[asset]).value_counts()
        asset_regime_pcts[asset] = (counts / len(combined_regimes) * 100)
    
    for regime in ['bull', 'correction', 'bear', 'rebound']:
        if regime in asset_regime_pcts.index:
            fig.add_trace(
                go.Box(
                    y=asset_regime_pcts.loc[regime],
                    name=regime,
                    marker_color=colors[regime]
                ),
                row=2, col=1
            )
    
    # 4. Regime Transitions Heatmap
    transitions = pd.DataFrame()
    for asset in combined_regimes.columns:
        series = combined_regimes[asset]
        curr_transitions = pd.crosstab(
            pd.Series(series), 
            pd.Series(series.shift(1))
        )
        transitions = transitions.add(curr_transitions, fill_value=0)
    
    fig.add_trace(
        go.Heatmap(
            z=transitions.values,
            x=transitions.columns,
            y=transitions.index,
            colorscale='Viridis',
            text=transitions.values.round(2),
            texttemplate='%{text}',
            textfont={"size": 10},
        ),
        row=2, col=2
    )
    
    # Update layout
    fig.update_layout(
        height=800,
        width=1200,
        showlegend=True,
        title_text="RORO Model Analysis - Carteira AC"
    )
    
    fig.show()
    
    # Print summary statistics
    print("\nOverall Regime Distribution:")
    overall_dist = combined_regimes.values.flatten()
    overall_counts = pd.Series(overall_dist).value_counts()
    print((overall_counts / len(overall_dist) * 100).round(2))
    
    # Print current regimes for each asset
    print("\nCurrent Regimes by Asset:")
    current_regimes = combined_regimes.iloc[-1].sort_index()
    for asset, regime in current_regimes.items():
        print(f"{asset}: {regime}")

# Run the analysis
analyze_and_plot_regimes()

In [9]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from geckoAPI.assetsRoster import carteira_AC

def compute_momentum_regime(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute momentum-based regime classification with volatility timing
    """
    # Compute log returns
    returns = np.log(df).diff()
    
    # Compute momentum signals
    fast_momentum = returns.rolling(window=30).sum()  # 1-month momentum
    slow_momentum = returns.rolling(window=365).sum() # 12-month momentum
    
    # Compute 30-day realized volatility
    realized_vol = returns.rolling(window=30).std() * np.sqrt(252)  # Annualized
    
    # Define high volatility threshold (80th percentile of historical volatility)
    vol_threshold = realized_vol.rolling(window=365).quantile(0.8)
    
    # Convert to binary signals (+1 for positive, -1 for negative)
    fast_signal = np.sign(fast_momentum)
    slow_signal = np.sign(slow_momentum)
    
    # Initialize regime DataFrame
    regime = pd.DataFrame(index=df.index)
    
    # Classify regimes with volatility condition
    conditions = [
        # Bull: Strong uptrend (both fast and slow positive) + Low volatility
        (fast_signal == 1) & (slow_signal == 1) & (realized_vol <= vol_threshold),
        
        # Correction: Short-term weakness (fast negative, slow positive) + Low volatility
        (fast_signal == -1) & (slow_signal == 1) & (realized_vol <= vol_threshold),
        
        # Bear: Downtrend (both negative) OR High volatility
        ((fast_signal == -1) & (slow_signal == -1)) | (realized_vol > vol_threshold),
        
        # Rebound: Recovering trend (fast positive, slow negative) + Low volatility
        (fast_signal == 1) & (slow_signal == -1) & (realized_vol <= vol_threshold)
    ]
    
    choices = ['bull', 'correction', 'bear', 'rebound']
    regime['regime'] = np.select(conditions, choices, default='undefined')
    
    # Add metrics for reference
    regime['realized_volatility'] = realized_vol
    regime['volatility_threshold'] = vol_threshold
    regime['fast_momentum'] = fast_momentum
    regime['slow_momentum'] = slow_momentum
    
    return regime

def analyze_and_plot_regimes():
    """
    Analyze and plot regime distributions with volatility overlay
    """
    all_regimes = {}
    all_metrics = {}
    
    # Process each asset
    for asset in carteira_AC:
        try:
            # Load and clean data
            df = pd.read_csv(f'micro/assetData/{asset}.csv')
            df['date'] = pd.to_datetime(df['date'])
            df = df.sort_values('date').groupby('date').last().reset_index()
            df.set_index('date', inplace=True)
            
            # Compute regime and metrics
            regime_df = compute_momentum_regime(df.iloc[:, 0])
            all_regimes[asset] = regime_df['regime']
            all_metrics[asset] = {
                'vol': regime_df['realized_volatility'],
                'vol_threshold': regime_df['volatility_threshold'],
                'fast_momentum': regime_df['fast_momentum'],
                'slow_momentum': regime_df['slow_momentum']
            }
            
        except Exception as e:
            print(f"Error processing {asset}: {str(e)}")
    
    # Combine all regimes
    combined_regimes = pd.DataFrame(all_regimes)
    combined_regimes = combined_regimes.loc[~combined_regimes.index.duplicated(keep='last')]
    
    # Create plots
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Regime Distribution Over Time',
            'Current Regime Distribution',
            'Asset Volatility Levels',
            'Regime Transitions'
        ),
        specs=[[{"type": "xy"}, {"type": "domain"}],
               [{"type": "xy"}, {"type": "xy"}]]
    )
    
    # Define colors
    colors = {
        'bull': '#03C04A',     # Green
        'correction': '#FFA500', # Orange
        'bear': '#FF0000',      # Red
        'rebound': '#0492C2'    # Blue
    }
    
    # 1. Regime Distribution Over Time
    regime_counts = pd.DataFrame()
    for col in combined_regimes.columns:
        counts = pd.Series(combined_regimes[col]).value_counts()
        regime_counts[col] = counts
    
    regime_percentages = regime_counts.div(regime_counts.sum(axis=0), axis=1) * 100
    
    for regime in ['bull', 'correction', 'bear', 'rebound']:
        if regime in regime_percentages.index:
            fig.add_trace(
                go.Scatter(
                    x=regime_percentages.columns,
                    y=regime_percentages.loc[regime],
                    name=regime,
                    mode='lines',
                    line=dict(color=colors[regime]),
                    stackgroup='one'
                ),
                row=1, col=1
            )
    
    # 2. Current Regime Distribution (Pie Chart)
    latest_regimes = pd.Series(combined_regimes.iloc[-1]).value_counts()
    fig.add_trace(
        go.Pie(
            labels=latest_regimes.index,
            values=latest_regimes.values,
            marker_colors=[colors.get(r, '#GRAY') for r in latest_regimes.index]
        ),
        row=1, col=2
    )
    
    # 3. Volatility Plot
    for asset in all_metrics:
        fig.add_trace(
            go.Scatter(
                x=all_metrics[asset]['vol'].index,
                y=all_metrics[asset]['vol'],
                name=f"{asset} Vol",
                line=dict(width=1),
                opacity=0.5
            ),
            row=2, col=1
        )
        fig.add_trace(
            go.Scatter(
                x=all_metrics[asset]['vol_threshold'].index,
                y=all_metrics[asset]['vol_threshold'],
                name=f"{asset} Threshold",
                line=dict(dash='dash', width=1),
                opacity=0.3
            ),
            row=2, col=1
        )
    
    # 4. Regime Transitions Heatmap
    transitions = pd.DataFrame()
    for asset in combined_regimes.columns:
        series = combined_regimes[asset]
        curr_transitions = pd.crosstab(
            pd.Series(series), 
            pd.Series(series.shift(1))
        )
        transitions = transitions.add(curr_transitions, fill_value=0)
    
    fig.add_trace(
        go.Heatmap(
            z=transitions.values,
            x=transitions.columns,
            y=transitions.index,
            colorscale='Viridis',
            text=transitions.values.round(2),
            texttemplate='%{text}',
            textfont={"size": 10},
        ),
        row=2, col=2
    )
    
    # Update layout
    fig.update_layout(
        height=800,
        width=1200,
        showlegend=True,
        title_text="RORO Model Analysis - Carteira AC with Volatility Timing"
    )
    
    fig.show()
    
    # Define positioning advice
    positioning_map = {
        'bull': 'BUY - Fully Long Risk Assets',
        'correction': 'REDUCE - Partial Longs',
        'bear': 'SELL - Defensive Positioning',
        'rebound': 'GRADUAL BUY - Re-Entry Phase'
    }
    
    # Print summary statistics
    print("\nOverall Regime Distribution:")
    overall_dist = combined_regimes.values.flatten()
    overall_counts = pd.Series(overall_dist).value_counts()
    print((overall_counts / len(overall_dist) * 100).round(2))
    
    # Print current market analysis
    print("\nCurrent Market Analysis:")
    current_regimes = combined_regimes.iloc[-1].sort_index()
    
    for asset, regime in current_regimes.items():
        vol = all_metrics[asset]['vol'].iloc[-1]
        vol_threshold = all_metrics[asset]['vol_threshold'].iloc[-1]
        fast_mom = all_metrics[asset]['fast_momentum'].iloc[-1]
        slow_mom = all_metrics[asset]['slow_momentum'].iloc[-1]
        
        print(f"\n{asset.upper()}:")
        print(f"Regime: {regime}")
        print(f"Current Volatility: {vol:.2%}")
        print(f"Volatility Threshold: {vol_threshold:.2%}")
        print(f"Fast Momentum: {fast_mom:.2%}")
        print(f"Slow Momentum: {slow_mom:.2%}")
        print(f"Positioning: {positioning_map[regime]}")

# Run the analysis
analyze_and_plot_regimes()


Overall Regime Distribution:
bear          11.74
bull           9.21
undefined      8.07
correction     6.43
rebound        4.06
Name: count, dtype: float64

Current Market Analysis:

AAVE:
Regime: correction
Current Volatility: 95.35%
Volatility Threshold: 98.87%
Fast Momentum: -15.99%
Slow Momentum: 100.64%
Positioning: REDUCE - Partial Longs

BITCOIN:
Regime: bull
Current Volatility: 34.14%
Volatility Threshold: 49.51%
Fast Momentum: 4.43%
Slow Momentum: 71.70%
Positioning: BUY - Fully Long Risk Assets

BLOCKSTACK:
Regime: bear
Current Volatility: 87.00%
Volatility Threshold: 96.83%
Fast Momentum: -49.72%
Slow Momentum: -68.58%
Positioning: SELL - Defensive Positioning

CHAINLINK:
Regime: bear
Current Volatility: 80.20%
Volatility Threshold: 80.55%
Fast Momentum: -6.16%
Slow Momentum: -0.05%
Positioning: SELL - Defensive Positioning

ETHENA:
Regime: undefined
Current Volatility: 131.44%
Volatility Threshold: nan%
Fast Momentum: -54.32%
Slow Momentum: nan%


KeyError: 'undefined'

In [29]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from typing import Tuple

def compute_position_signal(asset: str) -> Tuple[float, pd.Series, dict]:
    """
    Compute position signal (1.0, 0.5, or 0.0) based on momentum and volatility regime
    """
    try:
        # Load and clean data
        df = pd.read_csv(f'micro/assetData/{asset}.csv')
        df['date'] = pd.to_datetime(df['date'])
        df = df[df['date'] >= '2017-01-01']  # Filter for data from 2017 onwards
        df = df.sort_values('date').groupby('date').last().reset_index()
        df.set_index('date', inplace=True)
        
        # Compute returns
        price = df.iloc[:, 0]  # First column is price
        returns = np.log(price).diff()
        
        # Skip if not enough data
        if len(returns) < 365:  # Need at least 1 year of data
            return 0.5, pd.Series(), {}
        
        # Compute signals
        fast_momentum = returns.rolling(window=30).sum()  # 1-month momentum
        slow_momentum = returns.rolling(window=365).sum() # 12-month momentum
        realized_vol = returns.rolling(window=30).std() * np.sqrt(252)  # 30-day annualized vol
        vol_threshold = realized_vol.rolling(window=365).quantile(0.8)
        
        # Convert to binary signals
        fast_signal = np.sign(fast_momentum)
        slow_signal = np.sign(slow_momentum)
        
        # Initialize position series
        positions = pd.Series(index=df.index, dtype=float)
        
        # Compute positions based on regime
        for i in range(len(df)):
            if i < 365:  # Not enough history
                positions.iloc[i] = 0.5
                continue
                
            curr_fast = fast_signal.iloc[i]
            curr_slow = slow_signal.iloc[i]
            curr_vol = realized_vol.iloc[i]
            curr_vol_thresh = vol_threshold.iloc[i]
            
            # Bull: Strong uptrend + Low vol -> Full position
            if curr_fast == 1 and curr_slow == 1 and curr_vol <= curr_vol_thresh:
                positions.iloc[i] = 1.0
            
            # Correction: Short-term weakness + Low vol -> Half position
            elif curr_fast == -1 and curr_slow == 1 and curr_vol <= curr_vol_thresh:
                positions.iloc[i] = 0.5
            
            # Bear: Downtrend or High vol -> No position
            elif (curr_fast == -1 and curr_slow == -1) or curr_vol > curr_vol_thresh:
                positions.iloc[i] = 0.0
            
            # Rebound: Recovering trend + Low vol -> Half position
            elif curr_fast == 1 and curr_slow == -1 and curr_vol <= curr_vol_thresh:
                positions.iloc[i] = 0.5
            
            else:  # Undefined -> Half position
                positions.iloc[i] = 0.5
        
        # Get current metrics
        current_metrics = {
            'fast_momentum': fast_momentum.iloc[-1],
            'slow_momentum': slow_momentum.iloc[-1],
            'volatility': realized_vol.iloc[-1],
            'vol_threshold': vol_threshold.iloc[-1],
            'price': price.iloc[-1],
            'returns': returns.iloc[-1]
        }
        
        return positions.iloc[-1], positions, current_metrics
        
    except Exception as e:
        print(f"Error processing {asset}: {str(e)}")
        return 0.5, pd.Series(), {}  # Return neutral position if error

def plot_strategy_returns(asset: str, initial_investment: float = 10000):
    """
    Plot dollar value of investment with and without position signals
    
    Parameters:
    -----------
    asset : str
        Asset symbol to analyze
    initial_investment : float
        Initial investment amount in dollars (default: $10,000)
    """
    # Get position signals
    current_pos, positions, metrics = compute_position_signal(asset)
    
    # Load price data
    df = pd.read_csv(f'micro/assetData/{asset}.csv')
    df['date'] = pd.to_datetime(df['date'])
    df = df[df['date'] >= '2017-01-01']  # Filter for data from 2017 onwards
    df = df.sort_values('date').groupby('date').last().reset_index()
    df.set_index('date', inplace=True)
    
    # Compute returns
    returns = np.log(df.iloc[:, 0]).diff()
    
    # Compute dollar values
    buy_hold_value = initial_investment * (1 + returns).cumprod()
    strategy_value = initial_investment * (1 + returns * positions).cumprod()
    
    # Create plot
    fig = go.Figure()
    
    # Add buy & hold value
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=buy_hold_value,
            name='Buy & Hold',
            line=dict(color='gray', width=1)
        )
    )
    
    # Add strategy value
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=strategy_value,
            name='Strategy',
            line=dict(color='blue', width=2)
        )
    )
    
    # Update layout
    fig.update_layout(
        title=f"{asset.upper()} - ${initial_investment:,.0f} Investment Growth (2017-Present)",
        yaxis_title="Portfolio Value ($)",
        yaxis_tickformat="$,.0f",
        height=600,
        width=1000,
        showlegend=True
    )
    
    fig.show()
    
    # Print current analysis
    print(f"\nCurrent Analysis for {asset.upper()}:")
    print(f"Position Signal: {current_pos:.1f}")
    print(f"Current Volatility: {metrics['volatility']:.2%}")
    print(f"Volatility Threshold: {metrics['vol_threshold']:.2%}")
    print(f"Fast Momentum: {metrics['fast_momentum']:.2%}")
    print(f"Slow Momentum: {metrics['slow_momentum']:.2%}")
    
    # Compute performance metrics
    final_strategy_value = strategy_value.iloc[-1]
    final_buy_hold_value = buy_hold_value.iloc[-1]
    
    print(f"\nPerformance Summary:")
    print(f"Initial Investment: ${initial_investment:,.2f}")
    print(f"Strategy Final Value: ${final_strategy_value:,.2f}")
    print(f"Buy & Hold Final Value: ${final_buy_hold_value:,.2f}")
    print(f"Strategy Outperformance: ${(final_strategy_value - final_buy_hold_value):,.2f}")
    print(f"Strategy Return: {((final_strategy_value/initial_investment - 1) * 100):.1f}%")
    print(f"Buy & Hold Return: {((final_buy_hold_value/initial_investment - 1) * 100):.1f}%")

# Add these functions after your existing code:

def backtest_strategy(asset: str, initial_investment: float = 10000) -> dict:
    """
    Backtest the momentum strategy and calculate performance metrics
    """
    # Get position signals
    current_pos, positions, _ = compute_position_signal(asset)
    
    # Load price data
    df = pd.read_csv(f'micro/assetData/{asset}.csv')
    df['date'] = pd.to_datetime(df['date'])
    df = df[df['date'] >= '2017-01-01']
    df = df.sort_values('date').groupby('date').last().reset_index()
    df.set_index('date', inplace=True)
    
    # Compute returns
    returns = np.log(df.iloc[:, 0]).diff()
    
    # Calculate strategy returns
    strategy_rets = returns * positions
    bh_rets = returns
    
    # Calculate cumulative values
    strategy_value = initial_investment * (1 + strategy_rets).cumprod()
    bh_value = initial_investment * (1 + bh_rets).cumprod()
    
    # Calculate daily returns for both strategies
    strategy_daily_rets = strategy_value.pct_change()
    bh_daily_rets = bh_value.pct_change()
    
    # Calculate metrics
    def calculate_metrics(returns, values):
        total_return = (values.iloc[-1] / values.iloc[0] - 1) * 100
        cagr = (values.iloc[-1] / values.iloc[0]) ** (252/len(returns)) - 1
        volatility = returns.std() * np.sqrt(252)
        sharpe = (cagr - 0.02) / volatility  # Assuming 2% risk-free rate
        max_drawdown = ((values / values.cummax() - 1)).min() * 100
        win_rate = (returns > 0).mean() * 100
        
        return {
            'Total Return (%)': total_return,
            'CAGR (%)': cagr * 100,
            'Volatility (%)': volatility * 100,
            'Sharpe Ratio': sharpe,
            'Max Drawdown (%)': max_drawdown,
            'Win Rate (%)': win_rate,
            'Final Value ($)': values.iloc[-1]
        }
    
    # Get metrics for both strategies
    strategy_metrics = calculate_metrics(strategy_daily_rets, strategy_value)
    bh_metrics = calculate_metrics(bh_daily_rets, bh_value)
    
    # Calculate additional strategy-specific metrics
    avg_position = positions.mean()
    position_changes = (positions.diff() != 0).sum()
    trades_per_year = position_changes / (len(positions) / 252)
    
    # Combine all metrics
    results = {
        'Asset': asset.upper(),
        'Period': f"{df.index[0].strftime('%Y-%m-%d')} to {df.index[-1].strftime('%Y-%m-%d')}",
        'Initial Investment': initial_investment,
        'Strategy Metrics': strategy_metrics,
        'Buy & Hold Metrics': bh_metrics,
        'Strategy Statistics': {
            'Average Position': avg_position,
            'Position Changes': position_changes,
            'Trades per Year': trades_per_year,
            'Current Position': current_pos
        }
    }
    
    return results

def create_metrics_table(results: dict) -> go.Figure:
    """
    Create a plotly table with backtest metrics
    """
    # Prepare data for the table
    metrics = ['Total Return (%)', 'CAGR (%)', 'Volatility (%)', 
              'Sharpe Ratio', 'Max Drawdown (%)', 'Win Rate (%)']
    
    strategy_values = [f"{results['Strategy Metrics'][m]:.2f}" for m in metrics]
    bh_values = [f"{results['Buy & Hold Metrics'][m]:.2f}" for m in metrics]
    
    # Create table
    fig = go.Figure(data=[go.Table(
        header=dict(
            values=['Metric', 'Strategy', 'Buy & Hold'],
            font=dict(size=12, color='white'),
            fill_color='darkblue',
            align=['left', 'center', 'center']
        ),
        cells=dict(
            values=[metrics, strategy_values, bh_values],
            font=dict(size=11),
            fill_color='lightgray',
            align=['left', 'center', 'center']
        )
    )])
    
    fig.update_layout(
        title=f"Performance Metrics - {results['Asset']}",
        height=400,
        width=800,
        margin=dict(t=50, l=0, r=0, b=0)
    )
    
    return fig

# Modify your plot_strategy_returns function to include the backtest table
def plot_strategy_returns(asset: str, initial_investment: float = 10000):
    """
    Plot strategy returns and backtest metrics
    """
    # Get position signals
    current_pos, positions, metrics = compute_position_signal(asset)
    
    # Load price data
    df = pd.read_csv(f'micro/assetData/{asset}.csv')
    df['date'] = pd.to_datetime(df['date'])
    df = df[df['date'] >= '2017-01-01']  # Filter for data from 2017 onwards
    df = df.sort_values('date').groupby('date').last().reset_index()
    df.set_index('date', inplace=True)
    
    # Compute returns
    returns = np.log(df.iloc[:, 0]).diff()
    
    # Compute dollar values
    buy_hold_value = initial_investment * (1 + returns).cumprod()
    strategy_value = initial_investment * (1 + returns * positions).cumprod()
    
    # Create plot
    fig = go.Figure()
    
    # Add buy & hold value
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=buy_hold_value,
            name='Buy & Hold',
            line=dict(color='gray', width=1)
        )
    )
    
    # Add strategy value
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=strategy_value,
            name='Strategy',
            line=dict(color='blue', width=2)
        )
    )
    
    # Update layout
    fig.update_layout(
        title=f"{asset.upper()} - ${initial_investment:,.0f} Investment Growth (2017-Present)",
        yaxis_title="Portfolio Value ($)",
        yaxis_tickformat="$,.0f",
        height=600,
        width=1000,
        showlegend=True
    )
    
    fig.show()
    
    # Create and show metrics table
    results = backtest_strategy(asset, initial_investment)
    metrics_table = create_metrics_table(results)
    metrics_table.show()
    
    # Print current analysis
    print(f"\nCurrent Analysis for {asset.upper()}:")
    print(f"Position Signal: {current_pos:.1f}")
    print(f"Current Volatility: {metrics['volatility']:.2%}")
    print(f"Volatility Threshold: {metrics['vol_threshold']:.2%}")
    print(f"Fast Momentum: {metrics['fast_momentum']:.2%}")
    print(f"Slow Momentum: {metrics['slow_momentum']:.2%}")
    
    # Add strategy statistics
    print("\nStrategy Statistics:")
    print(f"Average Position: {results['Strategy Statistics']['Average Position']:.2f}")
    print(f"Total Position Changes: {results['Strategy Statistics']['Position Changes']}")
    print(f"Trades per Year: {results['Strategy Statistics']['Trades per Year']:.1f}")

# Usage remains the same
asset = "chainlink"
plot_strategy_returns(asset, initial_investment=10000)



Current Analysis for CHAINLINK:
Position Signal: 0.0
Current Volatility: 80.20%
Volatility Threshold: 80.55%
Fast Momentum: -6.16%
Slow Momentum: -0.05%

Strategy Statistics:
Average Position: 0.47
Total Position Changes: 215
Trades per Year: 20.4


# Random Forest


In [13]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
import pandas_ta as ta
import plotly.graph_objects as go
from plotly.subplots import make_subplots

class CryptoTradingModel:
    def __init__(self, asset_id='bitcoin', prediction_horizon=5, target_return=0.02):
        """Initialize as before"""
        self.asset_id = asset_id
        self.prediction_horizon = prediction_horizon
        self.target_return = target_return
        self.model = None
        self.scaler = StandardScaler()
    
    def prepare_data(self):
        """Prepare feature data for modeling"""
        # Load price data
        candle_file = f'micro/candleData/{self.asset_id}_candles.csv'
        df = pd.read_csv(candle_file)
        df['date'] = pd.to_datetime(df['date'])
        df.set_index('date', inplace=True)
        
        # Calculate log returns
        df['log_return'] = np.log(df['close']).diff()
        
        # Calculate moving averages using pandas_ta
        df['ma30'] = ta.sma(df['close'], length=30)
        df['ma365'] = ta.sma(df['close'], length=365)
        
        # Calculate return periods
        for period in [7, 30, 90, 365]:
            df[f'return_{period}d'] = df['close'].pct_change(period)
        
        # Add day of week
        df['day_of_week'] = df.index.dayofweek
        
        # Calculate ATR using pandas_ta
        df['atr'] = ta.atr(
            high=df['high'],
            low=df['low'],
            close=df['close'],
            length=14
        )
        
        # Add technical indicators
        df['rsi'] = ta.rsi(df['close'], length=14)
        df['macd'] = ta.macd(df['close'])['MACD_12_26_9']
        
        # Fix Bollinger Bands calculation
        bbands = ta.bbands(df['close'])
        df['bb_upper'] = bbands['BBU_5_2.0']
        df['bb_middle'] = bbands['BBM_5_2.0']
        df['bb_lower'] = bbands['BBL_5_2.0']
        
        # Fix Stochastic calculation
        stoch = ta.stoch(df['high'], df['low'], df['close'])
        df['stoch'] = stoch['STOCHk_14_3_3']
        
        # Load macro data
        vix = pd.read_csv('macro/fredData/vix.csv')
        credit_spreads = pd.read_csv('macro/fredData/creditSpreadsHighGrade.csv')
        dxy = pd.read_csv('macro/fredData/dxy.csv')
        move = pd.read_csv('macro/fredData/move.csv')
        
        # Process macro data
        for macro_df in [vix, credit_spreads, dxy, move]:
            macro_df['date'] = pd.to_datetime(macro_df['date'])
            macro_df.set_index('date', inplace=True)
        
        # Merge macro data
        df['vix'] = vix['vix']
        df['credit_spreads'] = credit_spreads['creditSpreadsHighGrade']
        df['dxy'] = dxy['dxy']
        df['move'] = move['move']
        
        # Forward fill missing values
        df = df.fillna(method='ffill')
        
        # Create target variable
        future_returns = df['close'].pct_change(self.prediction_horizon).shift(-self.prediction_horizon)
        df['target'] = (future_returns > self.target_return).astype(int)
        
        # Select features
        feature_cols = [
            'log_return', 'ma30', 'ma365', 
            'return_7d', 'return_30d', 'return_90d', 'return_365d',
            'day_of_week', 'atr', 'rsi', 'macd', 
            'bb_upper', 'bb_middle', 'bb_lower', 'stoch',
            'vix', 'credit_spreads', 'dxy', 'move'
        ]
        
        # Normalize features
        df[feature_cols] = self.scaler.fit_transform(df[feature_cols])
        
        return df, feature_cols
    
    def optimize_hyperparameters(self, X_train, y_train):
        """Grid search for GradientBoostingClassifier hyperparameters"""
        param_grid = {
            'n_estimators': [100, 200, 300],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [3, 4, 5],
            'min_samples_split': [2, 5, 10],
            'subsample': [0.8, 0.9, 1.0]
        }
        
        grid_search = GridSearchCV(
            GradientBoostingClassifier(random_state=42),
            param_grid,
            cv=5,
            scoring='accuracy',
            n_jobs=-1
        )
        
        grid_search.fit(X_train, y_train)
        return grid_search.best_params_
    
    def train_model(self, train_start='2015-01-01', test_start='2023-01-01'):
        """Train the model"""
        # Prepare data
        df, feature_cols = self.prepare_data()
        
        # Split data
        train_data = df[train_start:test_start]
        test_data = df[test_start:]
        
        X_train = train_data[feature_cols]
        y_train = train_data['target']
        X_test = test_data[feature_cols]
        y_test = test_data['target']
        
        # Optimize hyperparameters
        best_params = self.optimize_hyperparameters(X_train, y_train)
        
        # Train model
        self.model = GradientBoostingClassifier(**best_params, random_state=42)
        self.model.fit(X_train, y_train)
        
        # Get predictions
        train_probs = self.model.predict_proba(X_train)[:, 1]
        test_probs = self.model.predict_proba(X_test)[:, 1]
        
        # Generate signals
        train_signals = np.where(train_probs > 0.7, 1, np.where(train_probs < 0.3, 0, 0.5))
        test_signals = np.where(test_probs > 0.7, 1, np.where(test_probs < 0.3, 0, 0.5))
        
        # Get feature importance
        feature_importance = pd.DataFrame({
            'feature': feature_cols,
            'importance': self.model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        return {
            'train_data': train_data,
            'test_data': test_data,
            'train_signals': train_signals,
            'test_signals': test_signals,
            'feature_importance': feature_importance,
            'feature_cols': feature_cols
        }

In [18]:
model = CryptoTradingModel(
    asset_id='bitcoin',
    prediction_horizon=5,  # 5-day prediction window
    target_return=0.02    # 2% target return threshold
)

# Train the model with specific date ranges
results = model.train_model(
    train_start='2016-01-01',
    test_start='2023-01-01'  # Everything after this date is out-of-sample
)

/var/folders/p_/s5m9ntfs6rs0ntx6z7795qgrsb4pj6/T/ipykernel_47131/1387554095.py:81: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



In [19]:
def analyze_results(results):
    """Analyze model performance and display metrics"""
    try:
        # Create visualization
        fig = make_subplots(
            rows=3, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.05,
            subplot_titles=(
                'Price & Trading Signals',
                'Model Performance',
                'Feature Importance'
            ),
            row_heights=[0.5, 0.3, 0.2]
        )
        
        # Plot 1: Price and Trading Signals
        for period, data, signals in [
            ('Train', results['train_data'], results['train_signals']),
            ('Test', results['test_data'], results['test_signals'])
        ]:
            print(f"Processing {period} data...")
            
            # Convert signals to pandas Series with the same index as data
            signals_series = pd.Series(signals, index=data.index)
            
            # Plot price
            fig.add_trace(
                go.Scatter(
                    x=data.index,
                    y=data['close'],
                    name=f'{period} Price',
                    line=dict(color='lightgray', width=1)
                ),
                row=1, col=1
            )
            
            # Plot signals
            for signal_value, color in [(1, 'green'), (0, 'red')]:
                mask = signals_series == signal_value
                if any(mask):
                    fig.add_trace(
                        go.Scatter(
                            x=data.index[mask],
                            y=data.loc[mask, 'close'],
                            mode='markers',
                            name=f'{period} {"Buy" if signal_value == 1 else "Sell"}',
                            marker=dict(color=color, size=8)
                        ),
                        row=1, col=1
                    )
        
        # Plot 2: Cumulative Returns
        for period, data, signals in [
            ('Train', results['train_data'], results['train_signals']),
            ('Test', results['test_data'], results['test_signals'])
        ]:
            print(f"Calculating returns for {period}...")
            
            # Convert signals to pandas Series
            signals_series = pd.Series(signals, index=data.index)
            
            returns = data['close'].pct_change()
            strategy_returns = returns * signals_series.shift(1)
            
            cum_returns = (1 + returns).cumprod()
            cum_strategy = (1 + strategy_returns).cumprod()
            
            fig.add_trace(
                go.Scatter(
                    x=data.index,
                    y=cum_returns,
                    name=f'{period} Buy & Hold',
                    line=dict(color='gray', dash='dot')
                ),
                row=2, col=1
            )
            
            fig.add_trace(
                go.Scatter(
                    x=data.index,
                    y=cum_strategy,
                    name=f'{period} Strategy',
                    line=dict(color='blue')
                ),
                row=2, col=1
            )
        
        # Plot 3: Feature Importance
        print("Plotting feature importance...")
        importance_df = results['feature_importance'].head(10)
        fig.add_trace(
            go.Bar(
                x=importance_df['feature'],
                y=importance_df['importance'],
                name='Feature Importance'
            ),
            row=3, col=1
        )
        
        # Update layout
        fig.update_layout(
            height=1200,
            title_text='Trading Model Analysis',
            showlegend=True
        )
        
        fig.show()
        
        # Print performance metrics
        print("\nPerformance Metrics:")
        for period, data, signals in [
            ('Training Period', results['train_data'], results['train_signals']),
            ('Testing Period', results['test_data'], results['test_signals'])
        ]:
            # Convert signals to pandas Series
            signals_series = pd.Series(signals, index=data.index)
            
            returns = data['close'].pct_change()
            strategy_returns = returns * signals_series.shift(1)
            
            total_return = (1 + strategy_returns).prod() - 1
            annual_return = (1 + total_return) ** (252 / len(returns)) - 1
            sharpe = np.sqrt(252) * strategy_returns.mean() / strategy_returns.std()
            
            print(f"\n{period}:")
            print(f"Total Return: {total_return:.2%}")
            print(f"Annual Return: {annual_return:.2%}")
            print(f"Sharpe Ratio: {sharpe:.2f}")
            
            # Position statistics
            pos_changes = (signals_series.diff() != 0).sum()
            avg_position = signals_series.mean()
            trades_per_year = pos_changes * 252 / len(signals_series)
            
            print("\nTrading Statistics:")
            print(f"Average Position: {avg_position:.2f}")
            print(f"Total Trades: {pos_changes}")
            print(f"Trades per Year: {trades_per_year:.1f}")
            
    except Exception as e:
        print(f"Error in analyze_results: {str(e)}")
        import traceback
        traceback.print_exc()

# Run the analysis
analyze_results(results)

Processing Train data...
Processing Test data...
Calculating returns for Train...
Calculating returns for Test...
Plotting feature importance...



Performance Metrics:

Training Period:
Total Return: 917.62%
Annual Return: 25.68%
Sharpe Ratio: 0.89

Trading Statistics:
Average Position: 0.50
Total Trades: 5
Trades per Year: 0.5

Testing Period:
Total Return: 151.47%
Annual Return: 35.23%
Sharpe Ratio: 1.71

Trading Statistics:
Average Position: 0.44
Total Trades: 47
Trades per Year: 15.4


URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1006)>